In [ ]:
# Clear CUDA cache to free up GPU memory
if __name__ != "__main__":
    import torch
    torch.cuda.empty_cache()
    print("✓ CUDA cache cleared")

NameError: name 'torch' is not defined

In [ ]:
"""
Sanskrit Scripture RAG (Retrieval-Augmented Generation) Assistant
================================================================================
This notebook implements a Q&A system for Sanskrit scriptures (Ramayana/Mahabharata)
using LangChain, FAISS vector store, and local fine-tuned LLM.

Key Features:
- FAISS-based semantic search over Sanskrit/English shlokas
- Local fine-tuned Qwen model for contextual responses
- Citation of source book, chapter, and shloka numbers
- Devanagari to Roman transliteration support
"""

import os
import sys
import torch 
import pandas as pd
import logging
from pathlib import Path

# Configure logging for better debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# LangChain and Hugging Face imports
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_core.documents import Document
from peft import PeftModel

# Transliteration imports
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# ============================================================================
# CONFIGURATION - Centralized settings
# ============================================================================
class Config:
    """Configuration class for all paths and model parameters"""
    # Paths
    EMBEDDINGS_MODEL = "intfloat/multilingual-e5-large"
    BASE_MODEL = "qwen"
    ADAPTER_PATH = "qwen-finetuned"
    FULL_MODEL_PATH = "qwen7-finetuned"
    CSV_PATH = "res/combined.csv"
    FAISS_INDEX_PATH = "sanskrit_faiss_index_itihasa"
    
    # Model parameters
    MAX_TOKENS = 512
    TEMPERATURE = 0.01
    TOP_P = 0.9
    RETRIEVER_K = 2  # Number of documents to retrieve
    
    # Quantization config
    @staticmethod
    def get_bnb_config():
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16 
        )


# ============================================================================
# 0. UTILITY FUNCTIONS
# ============================================================================

def devanagari_to_roman(devanagari_text: str, output_scheme=sanscript.IAST) -> str:
    """
    Transliterate Devanagari text to Roman script.
    
    Args:
        devanagari_text: Text in Devanagari script
        output_scheme: Target transliteration scheme (default: IAST)
    
    Returns:
        Romanized text or error message
    """
    try:
        romanized = transliterate(devanagari_text, sanscript.DEVANAGARI, output_scheme)
        return romanized
    except Exception as e:
        logger.error(f"Transliteration failed: {e}")
        return f"Error during transliteration: {e}"


def load_and_chunk_itihasa_data(csv_path: str = "combined.csv") -> list:
    """
    Load CSV data and create LangChain Documents.
    
    Each shloka becomes one Document with metadata containing:
    - book, chapter, shloka_num, sanskrit_text
    
    Args:
        csv_path: Path to CSV file containing shloka data
    
    Returns:
        List of LangChain Documents
    """
    try:
        if not os.path.exists(csv_path):
            logger.warning(f"CSV not found at {csv_path}, creating sample documents")
            return []
        
        df = pd.read_csv(csv_path)
        logger.info(f"Loaded CSV with {len(df)} rows")
        
        documents = []
        for index, row in df.iterrows():
            # Combine Sanskrit and English for rich semantic embeddings
            content = f"Sanskrit Shloka: {row['sn']}\nEnglish Translation: {row['en']}"
            
            # Preserve metadata for citation
            metadata = {
                "book": row.get('book', 'Unknown'),
                "chapter": row.get('chapter', 'Unknown'),
                "shloka_num": row.get('shloka', 'Unknown'),
                "sanskrit_text": row['sn']
            }
            documents.append(Document(page_content=content, metadata=metadata))
        
        logger.info(f"Created {len(documents)} documents from CSV")
        return documents
    
    except Exception as e:
        logger.error(f"Error loading CSV data: {e}")
        return []


# ============================================================================
# 1. INITIALIZE EMBEDDINGS AND FAISS VECTOR STORE
# ============================================================================

def initialize_vector_store(chunk_docs=None):
    """
    Load or create FAISS vector store with HuggingFace embeddings.
    
    Args:
        chunk_docs: Optional list of documents to create new index from
    
    Returns:
        FAISS vector store instance
    """
    logger.info(f"Initializing vector store from: {Config.FAISS_INDEX_PATH}")
    
    # Initialize embedding model
    embedding_model = HuggingFaceEmbeddings(model_name=Config.EMBEDDINGS_MODEL)
    
    if os.path.exists(Config.FAISS_INDEX_PATH):
        logger.info(f"Loading existing FAISS index from disk")
        vector_store = FAISS.load_local(
            folder_path=Config.FAISS_INDEX_PATH, 
            embeddings=embedding_model, 
            allow_dangerous_deserialization=True
        )
    else:
        logger.info("FAISS index not found, creating new index...")
        if chunk_docs is None:
            chunk_docs = load_and_chunk_itihasa_data(Config.CSV_PATH)
        
        if not chunk_docs:
            logger.warning("No documents available for FAISS index creation")
            return None
        
        vector_store = FAISS.from_documents(chunk_docs, embedding_model)
        vector_store.save_local(Config.FAISS_INDEX_PATH)
        logger.info(f"New FAISS index saved to {Config.FAISS_INDEX_PATH}")
    
    return vector_store


# ============================================================================
# 2. SETUP LOCAL FINE-TUNED LLM
# ============================================================================

def setup_local_llm(use_adapter: bool = False) -> HuggingFacePipeline:
    """
    Initialize local fine-tuned LLM with graceful fallback.
    
    Args:
        use_adapter: If True, load adapter from ADAPTER_PATH, else load full model
    
    Returns:
        HuggingFacePipeline instance
    """
    model_path = Config.ADAPTER_PATH if use_adapter else Config.FULL_MODEL_PATH
    
    try:
        logger.info(f"Loading model from: {model_path}")
        
        bnb_config = Config.get_bnb_config()
        
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Load model
        if use_adapter:
            logger.info(f"Loading base model: {Config.BASE_MODEL}")
            base_model = AutoModelForCausalLM.from_pretrained(
                Config.BASE_MODEL, 
                quantization_config=bnb_config, 
                device_map="auto"
            )
            model = PeftModel.from_pretrained(base_model, Config.ADAPTER_PATH)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_path, 
                quantization_config=bnb_config, 
                device_map="auto"
            )
        
        model.eval()
        logger.info(f"Successfully loaded model from {model_path}")
        
    except Exception as e:
        logger.warning(f"Failed to load fine-tuned model: {e}")
        logger.info("Falling back to distilgpt2 for testing")
        tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
        model = AutoModelForCausalLM.from_pretrained("distilgpt2")
    
    # Create pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=Config.MAX_TOKENS,
        temperature=Config.TEMPERATURE,
        do_sample=True,
        top_p=Config.TOP_P,
        return_full_text=False
    )

    llm = HuggingFacePipeline(pipeline=pipe)
    logger.info("HuggingFace pipeline initialized")
    return llm


# ============================================================================
# 3. SETUP RAG CHAIN
# ============================================================================

def setup_rag_chain(llm: HuggingFacePipeline, vector_store: FAISS):
    """
    Create RetrievalQA chain combining retriever and LLM.
    
    Args:
        llm: Language model instance
        vector_store: FAISS vector store for retrieval
    
    Returns:
        RetrievalQA chain instance
    """
    
    # Define system prompt with citation requirements
    qa_prompt = PromptTemplate(
        template="""You are an expert Sanskrit scripture research assistant.
        
Use the following retrieved context to answer the question accurately and concisely.

Context:
{context}

Question:
{question}

Instructions:
1. Provide a clear, informative answer in English
2. If relevant, cite the source (book, chapter, shloka number) 
3. If the context doesn't contain relevant information, say so clearly
4. Be precise and avoid speculation""",
        input_variables=["context", "question"],
    )
    
    logger.info("Creating RetrievalQA chain")
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff", 
        retriever=vector_store.as_retriever(k=Config.RETRIEVER_K), 
        chain_type_kwargs={"prompt": qa_prompt},
    )
    
    return qa_chain


# ============================================================================
# 4. INTERACTIVE CHAT INTERFACE
# ============================================================================

def chat_with_assistant(qa_chain: RetrievalQA):
    """
    Interactive chat loop for user queries.
    
    Args:
        qa_chain: RetrievalQA chain instance
    """
    print("\n" + "="*70)
    print("Sanskrit Scripture Research Assistant")
    print("="*70)
    print("\nType 'quit' or 'exit' to end the conversation.\n")
    
    while True:
        try:
            user_query = input("\n📚 Your Question: ").strip()
            
            if not user_query:
                print("Please enter a question.")
                continue
            
            if user_query.lower() in ["quit", "exit"]:
                print("\n✅ Thank you for using the Sanskrit Research Assistant. Goodbye!")
                break
            
            # Get response from RAG chain
            logger.info(f"Processing query: {user_query[:50]}...")
            result = qa_chain.invoke({"query": user_query})
            
            print("\n" + "-"*70)
            print("📖 Answer:")
            print("-"*70)
            print(result['result'])
            
        except KeyboardInterrupt:
            print("\n\n✅ Interrupted. Exiting assistant.")
            break
        except Exception as e:
            logger.error(f"Error processing query: {e}")
            print(f"\n⚠️ An error occurred: {e}")


# ============================================================================
# 5. MAIN EXECUTION
# ============================================================================

def main():
    """Main entry point for the application"""
    logger.info("Starting Sanskrit Scripture RAG Assistant")
    
    # Step 1: Initialize vector store
    chunks = load_and_chunk_itihasa_data(Config.CSV_PATH)
    vector_store = initialize_vector_store(chunks)
    
    if vector_store is None:
        logger.error("Failed to initialize vector store. Exiting.")
        return
    
    logger.info(f"Vector store ready with {len(chunks)} documents")
    
    # Step 2: Setup LLM
    llm = setup_local_llm(use_adapter=False)
    
    # Step 3: Create RAG chain
    qa_chain = setup_rag_chain(llm, vector_store)
    logger.info("RAG chain initialized")
    
    # Step 4: Start interactive chat
    chat_with_assistant(qa_chain)


if __name__ == "__main__":
    main()
